Activation Functions: The Essentials

Implement and compare all major activation functions




In [12]:
import numpy as np

# activation functions from scratch

def relu(x):
  # relu: max(0,x) - blocks negatives, passes positives
  return np.maximum(0,x)

def leaky_relu(x, alpha=0.01):
  # leaky relu: small slope for negatives prevents "dying neurons"
  return np.where(x > 0, x, alpha * x)

def sigmoid(x):
  # Sigmoid: squashes to 0,1 - for probabilities
  return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def softmax(x):
  # softmax outputs sum to 1. - for multiclass probabilities
  exp_x = np.exp(x - np.max(x)) # Subtract max for numerical stability
  return exp_x / exp_x.sum()

def gelu(x):
  # gelu used in transformers (bert, gpt)
  return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

x = np.array([-2,-1, 0, 1, 2])
print("Input: ", x)
print("ReLU: ", relu(x))
print("Leaky ReLU: ", leaky_relu(x))
print("Sigmoid: ", sigmoid(x))
print("Softmax: ", softmax(x))
print("GeLU: ", gelu(x))

print("\n--- Softmax: Scores → Probabilities ---")
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print(f"Logits:  {logits}")
print(f"Softmax: {np.round(probs, 3)}")
print(f"Sum:     {probs.sum():.4f} (always 1.0)")

# why linearity collapses

print("\n -- Why we need activations---")
W1 = np.array([[1,2], [3,4]])
W2 = np.array([[5,6], [7,8]])
x = np.array([1,1])

# two linear layers
y_two_layers = W2 @ (W1 @ x)

# collapsed single layer
W_collapsed = W2 @ W1
y_one_layer = W_collapsed @ x

print(f"Two linear layers:    {y_two_layers}")
print(f"One collapsed layer:  {y_one_layer}")
print(f"Same result! Without activation, depth is useless.")


Input:  [-2 -1  0  1  2]
ReLU:  [0 0 0 1 2]
Leaky ReLU:  [-0.02 -0.01  0.    1.    2.  ]
Sigmoid:  [0.11920292 0.26894142 0.5        0.73105858 0.88079708]
Softmax:  [0.01165623 0.03168492 0.08612854 0.23412166 0.63640865]
GeLU:  [-0.04540231 -0.15880801  0.          0.84119199  1.95459769]

--- Softmax: Scores → Probabilities ---
Logits:  [2.  1.  0.1]
Softmax: [0.659 0.242 0.099]
Sum:     1.0000 (always 1.0)

 -- Why we need activations---
Two linear layers:    [57 77]
One collapsed layer:  [57 77]
Same result! Without activation, depth is useless.


Choosing the Right Activation

Build networks with appropriate activations for each task

In [13]:
import torch
import torch.nn as nn

# ============================================================
# ACTIVATION CHOICE BY TASK
# ============================================================

# binary classification : spam detection
# output - probability of class 1
class SpamClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(100,64), nn.ReLU(),
        nn.Linear(64, 1),
        nn.Sigmoid() # output probability- 0,1
    )
  def forward(self, x):
    return self.net(x)

# multiclass classfication  (image recognition eg. cat or dog, or cow, mouse)
# output - probability for each class (sum to 1)
class ImageClassifier(nn.Module):
  def __init__(self, num_of_classes=10):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256,128), nn.ReLU(),
        nn.Linear(128, num_of_classes)
        # No activation! CrossEntropyLoss includes softmax
    )
  def forward(self, x):
    return self.net(x)

# regression - house prices
# output - any real number
class HousePricePredictor(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(10,64), nn.ReLU(),
        nn.Linear(64,32), nn.ReLU(),
        nn.Linear(32,1)
        # no activation needed for linear
    )

  def forward(self,x):
    return self.net(x)

# transformer style - (gpt, bert)
# uses gelu in feed forward layers
class TransformeFNN(nn.Module):
  def __init__(self, d_model=768, d_ff = 3072):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(d_model,d_ff),
        nn.GELU(),
        nn.Linear(d_ff,d_model)
    )

  def forward(self, x):
    return self.net(x)



print("Activation Function Quick Reference")
print("=" * 50)
print("Hidden Layers:    ReLU (or GELU for Transformers)")
print("Binary Output:    Sigmoid → P(class=1)")
print("Multi-class:      Softmax → P(each class)")
print("Regression:       None (linear) → any real value")
print("=" * 50)

Activation Function Quick Reference
Hidden Layers:    ReLU (or GELU for Transformers)
Binary Output:    Sigmoid → P(class=1)
Multi-class:      Softmax → P(each class)
Regression:       None (linear) → any real value


In [14]:
print("\n--- Testing Model Forward Passes ---")

# Test SpamClassifier
spam_model = SpamClassifier()
sample_spam_input = torch.randn(1, 100) # Batch size 1, 100 features
spam_output = spam_model(sample_spam_input)
print(f"SpamClassifier output shape: {spam_output.shape}, value: {spam_output.item():.4f}")

# Test ImageClassifier
image_model = ImageClassifier(num_of_classes=10)
sample_image_input = torch.randn(1, 784) # Batch size 1, 784 pixels (e.g., 28x28)
image_output = image_model(sample_image_input)
print(f"ImageClassifier output shape: {image_output.shape}")

# Test HousePricePredictor
house_model = HousePricePredictor()
sample_house_input = torch.randn(1, 10) # Batch size 1, 10 features
house_output = house_model(sample_house_input)
print(f"HousePricePredictor output shape: {house_output.shape}, value: {house_output.item():.2f}")

# Test TransformerFNN
transformer_model = TransformeFNN(d_model=768, d_ff=3072)
sample_transformer_input = torch.randn(1, 768) # Batch size 1, d_model features
transformer_output = transformer_model(sample_transformer_input)
print(f"TransformerFNN output shape: {transformer_output.shape}")


--- Testing Model Forward Passes ---
SpamClassifier output shape: torch.Size([1, 1]), value: 0.4429
ImageClassifier output shape: torch.Size([1, 10])
HousePricePredictor output shape: torch.Size([1, 1]), value: 0.30
TransformerFNN output shape: torch.Size([1, 768])
